# 02 — Phân tích 216 run

Notebook này **không chạy thí nghiệm**. Nó đọc `results/summary.csv` và `results/raw/*.json`
đã có, rồi sinh mọi bảng và hình cho báo cáo cuối.

Để chạy lại thí nghiệm: `python src/run_all.py --phase all`

## 1. Nạp dữ liệu

In [ ]:
# Chuyển về thư mục gốc repo (notebook nằm trong notebooks/)
import os
if not os.path.exists('results/summary.csv') and os.path.exists('../results/summary.csv'):
    os.chdir('..')
print('cwd:', os.getcwd())
assert os.path.exists('results/summary.csv'), (
    'không thấy results/summary.csv — chạy notebook từ thư mục gốc repo')

In [ ]:
import sys, os, json, glob
sys.path.insert(0, 'src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.width', 200)
FIG = 'results/figures'
os.makedirs(FIG, exist_ok=True)
SEED = 42

df = pd.read_csv('results/summary.csv')
print('tổng run      :', len(df))
print('số cấu hình   :', df['name'].nunique())
vc = df['name'].value_counts()
bad = vc[vc != 3]
print('khác 3 seed   :', bad.to_string() if len(bad) else 'không có')
print('các pha       :', sorted(df['phase'].unique()))

In [ ]:
def raw(name, seed=SEED):
    """Đọc file JSON chi tiết của một run."""
    try:
        return json.load(open(f'results/raw/{name}_s{seed}.json'))
    except FileNotFoundError:
        return None

def with_depth(frame):
    """Tách tên cấu hình và độ sâu từ cột name."""
    d = frame[frame['name'].str.contains(r'_d\d+$', regex=True)].copy()
    d['cfg']   = d['name'].str.rsplit('_d', n=1).str[0]
    d['depth'] = d['name'].str.extract(r'_d(\d+)$').astype(int)
    return d

---
## 2. Pha 1 — Ablation cô lập từng biến (depth 7)

Mỗi cấu hình đổi **đúng một thành phần** so với baseline.

In [ ]:
p1 = df[df['phase'] == '1_ablation']
a = (p1.groupby('name')
       .agg(lr=('lr', 'mean'), acc=('test_acc', 'mean'), sd=('test_acc', 'std'),
            slope=('init_decay_slope', 'mean'), r2=('final_r2', 'mean'),
            drift_L1=('drift_L1', 'mean'), eff_depth=('init_eff_depth_0.01', 'mean'))
       .round(4))
print(a.sort_values('acc', ascending=False).to_string())

### Ba điểm đáng chú ý

1. `+adam` có `slope` **giống hệt** baseline nhưng accuracy 0.858 — nó không chạm vào dòng gradient
2. `R²` của baseline là 0.99 nên slope đáng tin; của `+adam` chỉ 0.15 nên slope mất ý nghĩa ở đó
3. `+he` có σ lớn gấp hàng chục lần các cấu hình khác — dấu hiệu của vùng chuyển tiếp

In [ ]:
print('Adam vs baseline — cùng kiến trúc, cùng init, chỉ khác optimizer:')
print(a.loc[['baseline', '+adam'],
            ['slope', 'eff_depth', 'drift_L1', 'acc']].to_string())
print('\nslope giống nhau + eff_depth giống nhau -> gradient KHÔNG được khôi phục')
print('drift_L1 cao hơn nhiều              -> Adam bù bằng bước cập nhật')

---
## 3. Pha 2 — Thứ hạng đổi theo độ sâu

In [ ]:
d = with_depth(df)
d = d[~d['cfg'].str.startswith(('base_w', 'fine_'))]
piv = d.pivot_table(index='cfg', columns='depth', values='test_acc', aggfunc='mean').round(3)
print(piv.to_string())

In [ ]:
print('=== Ngưỡng độ sâu (accuracy > 0.5) ===')
for cfg in piv.index:
    row = piv.loc[cfg].dropna()
    ok   = [x for x in row.index if row[x] > 0.5]
    fail = [x for x in row.index if row[x] <= 0.5]
    if ok and fail:
        print(f'{cfg:12s} sống đến d{max(ok):<3d} sập từ d{min(fail)}')
    elif ok:
        print(f'{cfg:12s} sống ở mọi depth đã thử (đến d{max(ok)})')

**Điểm quan trọng nhất:** ở depth 3 *mọi* cấu hình đều ~0.87. Nếu chỉ thí nghiệm
ở độ sâu nhỏ sẽ không phân biệt được gì. Khớp với He et al. 2015, vốn cũng quan sát
Xavier và He init gần như không khác nhau ở 14–22 layer nhưng tách hẳn ở 30 layer.

### Hình 1 — Accuracy theo độ sâu

In [ ]:
plt.figure(figsize=(9, 5.5))
for c, g in d.groupby('cfg'):
    m = g.groupby('depth')['test_acc'].agg(['mean', 'std']).sort_index()
    plt.errorbar(m.index, m['mean'], yerr=m['std'].fillna(0),
                 marker='o', capsize=3, label=c)
plt.axhline(0.1, ls='--', c='gray', lw=1, label='đoán ngẫu nhiên')
plt.xlabel('Số hidden layer'); plt.ylabel('Test accuracy')
plt.title('Ở mạng nông mọi giải pháp như nhau; phải sâu mới tách ra')
plt.legend(fontsize=8, ncol=2); plt.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f'{FIG}/fig1_depth_ranking.png', dpi=150); plt.show()

---
## 4. Một ngưỡng chung — kết luận trung tâm

`decay_slope` là mức suy giảm **mỗi layer**. Nhân với số layer để có **tổng suy giảm
toàn mạng**. Nếu đại lượng này là thứ quyết định, mọi cấu hình phải xếp theo nó.

In [ ]:
d['total'] = (d['init_decay_slope'] * d['depth']).round(2)
print('=== Tổng suy giảm (bậc 10) ===')
print(d.pivot_table(index='cfg', columns='depth', values='total', aggfunc='mean').round(2).to_string())

In [ ]:
g = (d.groupby('name')
       .agg(cfg=('cfg', 'first'), depth=('depth', 'first'),
            total=('total', 'mean'), acc=('test_acc', 'mean'))
       .reset_index())
g['opt'] = np.where(g['name'].str.contains('adam'), 'Adam', 'SGD')
g['hoc_duoc'] = g['acc'] > 0.5

for k in ['SGD', 'Adam']:
    s = g[g['opt'] == k]
    ok, bad = s[s['hoc_duoc']]['total'], s[~s['hoc_duoc']]['total']
    print(f'{k:5s}: học được tới {ok.max():.2f} | sập từ {bad.min():.2f}')

### Kiểm chứng bằng biến độc lập

Ngưỡng trên chỉ suy ra từ việc đổi `depth`. Nếu nó là quy luật thật thì đổi `width`
— một biến hoàn toàn khác — cũng phải tuân theo.

In [ ]:
w = df[df['name'].str.startswith('base_w')].copy()
w['width'] = w['name'].str.extract(r'_w(\d+)_').astype(int)
w['depth'] = w['name'].str.extract(r'_d(\d+)$').astype(int)
gw = (w.groupby(['width', 'depth'])
        .agg(acc=('test_acc', 'mean'), slope=('init_decay_slope', 'mean'))
        .reset_index())
gw['total'] = (gw['slope'] * gw['depth']).round(2)
gw['hoc_duoc'] = gw['acc'] > 0.5
print(gw.sort_values('total')[['width', 'depth', 'total', 'acc', 'hoc_duoc']].round(3).to_string(index=False))

ok, bad = gw[gw['hoc_duoc']]['total'], gw[~gw['hoc_duoc']]['total']
print(f'\nĐổi width: học được tới {ok.max():.2f} | sập từ {bad.min():.2f}')
print('Hai khoảng chồng nhau -> quy luật được kiểm chứng bằng biến độc lập.')

### Hình 2 — Ngưỡng tổng suy giảm *(hình trung tâm)*

In [ ]:
plt.figure(figsize=(9, 5.5))
for k, c, m in [('SGD', 'tab:blue', 'o'), ('Adam', 'tab:orange', 's')]:
    s = g[g['opt'] == k]
    sd = d[d['name'].isin(s['name'])].groupby('name')['test_acc'].std()
    plt.errorbar(s['total'], s['acc'], yerr=s['name'].map(sd).fillna(0),
                 fmt=m, color=c, capsize=3, label=k, alpha=0.85)
plt.axvspan(4.07, 4.32, color='tab:blue',   alpha=0.15, label='ngưỡng SGD ≈ 4.2')
plt.axvspan(8.06, 8.91, color='tab:orange', alpha=0.15, label='ngưỡng Adam ≈ 8.5')
plt.axhline(0.1, ls='--', c='gray', lw=1)
plt.xlabel('Tổng suy giảm gradient (bậc 10) = slope × depth')
plt.ylabel('Test accuracy')
plt.title('Một ngưỡng duy nhất; Adam nâng giới hạn lên gấp đôi')
plt.legend(fontsize=9); plt.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f'{FIG}/fig2_threshold.png', dpi=150); plt.show()

---
## 5. Ngoại lệ: `+relu` thất bại vì tín hiệu forward

`+relu` có tổng suy giảm **thấp nhất** — gần như không có vanishing gradient — nhưng vẫn sập.

In [ ]:
rows = []
for f in glob.glob('results/raw/+relu*_s42.json'):
    j = json.load(open(f)); c = j['config']; n = j.get('init_act_norm') or {}
    rows.append({'name': c['name'], 'depth': c['depth'],
                 'act_norm_L1': n.get('layer1'),
                 'act_norm_last': n.get(f"layer{c['depth']}")})
print(pd.DataFrame(rows).sort_values(['name', 'depth']).round(4).to_string(index=False))
print('\n+relu: act_norm ở layer cuối -> 0 khi depth tăng = tín hiệu FORWARD tắt hẳn.')
print('Nguyên nhân: init N(0, 0.05) quá nhỏ so với công thức He sqrt(2/128) ~ 0.125.')

### Hình 4 — Tín hiệu forward

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, cfg in zip(axes, ['+relu', '+relu_he']):
    for dep in [3, 7, 15, 25]:
        j = raw(f'{cfg}_d{dep}')
        if not j: continue
        n = j.get('init_act_norm') or {}
        ks = sorted(n, key=lambda s: int(s.replace('layer', '')))
        ax.plot(range(1, len(ks) + 1), [n[k] for k in ks], 'o-', label=f'depth {dep}')
    ax.set_yscale('log'); ax.set_xlabel('Layer index'); ax.set_title(cfg)
    ax.legend(fontsize=8); ax.grid(alpha=0.3, which='both')
axes[0].set_ylabel('RMS activation')
plt.suptitle('+relu thất bại vì tín hiệu FORWARD tắt, không phải vì gradient')
plt.tight_layout(); plt.savefig(f'{FIG}/fig4_forward_signal.png', dpi=150); plt.show()

---
## 6. Vùng chuyển tiếp lưỡng cực

Tại ngưỡng, kết quả không phải trung bình mà phụ thuộc may rủi của khởi tạo.

In [ ]:
t = df[df['name'] == 'fine_he_d7'][['seed', 'test_acc', 'init_decay_slope', 'final_r2', 'drift_L1']]
print(t.round(4).to_string(index=False))
print('\nHai seed thoát khỏi vanishing (R2 thấp, drift cao), một seed không.')

---
## 7. Ba chế độ của slope — trả lời phản biện về tính ổn định

In [ ]:
LABEL = {'fine_baseline_d7': 'baseline — không học (acc 0.10)',
         'fine_he_d7':       '+he — chuyển tiếp (acc ~0.38)',
         'fine_bn_d7':       '+bn — học tốt (acc 0.88)'}

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
for name, lab in LABEL.items():
    j = raw(name)
    if not j: print('thiếu', name); continue
    if j.get('slope_per_epoch'): ax1.plot(j['slope_per_epoch'], label=lab)
    if j.get('r2_per_epoch'):    ax2.plot(j['r2_per_epoch'],   label=lab)
ax1.set_ylabel('decay_slope (bậc/layer)')
ax1.set_title('Slope tĩnh ở hai thái cực, động ở vùng chuyển tiếp')
ax1.legend(fontsize=9); ax1.grid(alpha=0.3)
ax2.axhline(0.95, ls='--', c='gray', lw=1)
ax2.set_ylabel(r'$R^2$'); ax2.set_xlabel('Epoch'); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f'{FIG}/fig6_three_regimes.png', dpi=150); plt.show()

`+he` giữ slope ~0.52 đến epoch 33 rồi đổ xuống 0, R² sập cùng lúc — mạng **thoát khỏi
vanishing trong lúc train**. Khớp với quan sát của Glorot & Bengio rằng unit bão hòa
có thể tự thoát nhưng rất chậm, tạo plateau dài trước khi loss giảm tiếp.

---
## 8. Pha 3 — Leave-one-out

Bỏ từng giải pháp khỏi cấu hình đầy đủ. Dùng leave-one-out thay vì cộng dồn vì kết quả
cộng dồn phụ thuộc thứ tự thêm — giải pháp thêm sau luôn có vẻ đóng góp ít hơn.

In [ ]:
p3 = df[df['name'].str.startswith('full')]
b = p3.groupby('name')['test_acc'].agg(['mean', 'std']).round(5)
base, sd = b.loc['full', 'mean'], b.loc['full', 'std']
b['delta'] = (b['mean'] - base).round(5)
b['vuot_2sd'] = b['delta'].abs() > 2 * sd
print(b.sort_values('mean', ascending=False).to_string())
print(f'\nNgưỡng 2σ của full = {2*sd:.5f}')
print('Không vượt 2σ -> "chưa phân biệt được", tức giải pháp đó THỪA khi đã có các cái còn lại.')

### Hình 5 — Adam khuếch đại bước cập nhật

In [ ]:
plt.figure(figsize=(9, 5))
for name in ['baseline', '+adam']:
    j = raw(name)
    if not j: continue
    u = j['final_update_ratio']; ks = list(u.keys())
    plt.plot(ks, [u[k] for k in ks], 'o-', label=name)
plt.yscale('log'); plt.xticks(rotation=45)
plt.ylabel(r'$\|\Delta W\| / \|W\|$')
plt.title('Adam khuếch đại bước cập nhật ở layer có gradient nhỏ')
plt.legend(); plt.grid(alpha=0.3, which='both')
plt.tight_layout(); plt.savefig(f'{FIG}/fig5_update_ratio.png', dpi=150); plt.show()

### Hình 3 — Dòng gradient theo layer

In [ ]:
plt.figure(figsize=(9, 5.5))
for name in ['baseline', '+relu', '+he', '+relu_he', '+bn', '+residual', '+adam']:
    j = raw(name)
    if not j: continue
    r = j['init_grad_ratio']; ks = list(r.keys())
    plt.plot(ks, [r[k] for k in ks], 'o-', label=name)
plt.yscale('log'); plt.xticks(rotation=45)
plt.xlabel('Layer (input -> output)'); plt.ylabel(r'$\|\nabla W\|/\|W\|$')
plt.title('Dòng gradient tại khởi tạo, depth 7 (thang log)')
plt.legend(fontsize=9); plt.grid(alpha=0.3, which='both')
plt.tight_layout(); plt.savefig(f'{FIG}/fig3_grad_profile.png', dpi=150); plt.show()

---
## 9. Tóm tắt

| # | Phát hiện |
|---|---|
| 1 | Tổng suy giảm gradient ∈ [4.07, 4.32] bậc là ngưỡng quyết định với SGD, kiểm chứng bằng hai biến độc lập (depth và width) |
| 2 | Adam không chạm vào gradient (slope giống hệt baseline) nhưng nâng ngưỡng lên ~8.5 bậc |
| 3 | `+relu` thất bại vì tín hiệu **forward** tắt, không phải vì vanishing gradient |
| 4 | Tại ngưỡng, kết quả lưỡng cực và slope biến đổi theo thời gian; ngoài ngưỡng thì tĩnh |

Sáu hình đã lưu trong `results/figures/`.